# Step 6 — robustness to phone photos / WhatsApp
**Settings:** GPU T4 x2, Internet on, `HF_TOKEN`. Inputs: as before + Step 0-5 outputs; optional
`gajahterbang111/chexphoto-split` (UNOFFICIAL CheXphoto copy - check Stanford's licence first).
1) train the robust twin of the Step 5 winner (same recipe + 40% augmented images), 2) fixed photo-style test set,
3) both models on clean / photo / CheXphoto, 4) drop table.

In [ ]:
!git clone -q --depth 1 https://github.com/Vinith-44/Radioreport /kaggle/working/repo
%cd /kaggle/working/repo
!pip install -q -r requirements.txt

In [ ]:
import os, json
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

## 0. Restore the outputs of the earlier steps (add each previous notebook's output as an input)

In [ ]:
PREV = []   # e.g. ["/kaggle/input/step0-kaggle/project", "/kaggle/input/step1-kaggle/project", ...]
os.makedirs("/kaggle/working/project", exist_ok=True)
for src in PREV:
    !cp -rn {src}/outputs /kaggle/working/project/ 2>/dev/null || true
    !cp -rn {src}/data /kaggle/working/project/ 2>/dev/null || true
!ls /kaggle/working/project/outputs/runs

In [ ]:
WINNER = json.load(open("/kaggle/working/project/outputs/runs/step5/b2/winner.json"))["winner"]
print("Step 5 winner:", WINNER)
!python scripts/step0_prepare_data.py --config configs/step6.yaml

### 1. Robust twin (pick the cell that matches the winner)

In [ ]:
# Model A won:
# !python scripts/step1_train_qlora.py --config configs/step6_model_a.yaml --run-name r1
# Model B won:
# !torchrun --nproc_per_node 2 scripts/step4_train_projector.py --config configs/step6_model_b.yaml --run-name r1

### 2-4. Photo test set, generation, scores

In [ ]:
!python scripts/step6_make_photo_test.py
!python scripts/step6_eval.py --run-name e1
!python scripts/step6_score.py --run-name e1
!cat /kaggle/working/project/outputs/runs/step6/e1/results_table.md